# NodeSubstrates - Co-authorship Network Demo

This notebook demonstrates NodeSubstrates on a co-authorship network of visualization researchers,
as described in the paper. The dataset is derived from [vispubdata.org](https://vispubdata.org/).

In [17]:
from node_substrates import (
    NodeSubstratesWidget,
    load_coauthorship_sample,
    load_sample_network,
)

## 1. Load the Co-authorship Network

We load ~150 visualization researchers from IEEE VIS publications (2010-2024).
Each author has attributes including:
- **pub_count**: Number of publications
- **h_index_approx**: Approximate h-index
- **topic_***: Topic distribution from keywords
- **career_length**: Years active
- **degree**: Number of co-authors
- **clustering**: Local clustering coefficient
- **betweenness**: Betweenness centrality

In [18]:
# Load from vispubdata.org (requires internet)
# This downloads the data and builds the co-authorship network
try:
    G = load_coauthorship_sample(n_authors=500, min_papers=3, year_range=(2015, 2024))
except Exception as e:
    print(f"Could not download vispubdata: {e}")
    print("Using synthetic sample network instead...")
    G = load_sample_network()

print(f"\nNetwork: {G.number_of_nodes()} authors, {G.number_of_edges()} collaborations")
print(f"\nSample node attributes:")
sample_node = list(G.nodes())[0]
for key, value in G.nodes[sample_node].items():
    print(f"  {key}: {value}")

Loaded co-authorship network: 500 authors, 2078 collaborations

Network: 500 authors, 2078 collaborations

Sample node attributes:
  label: Benjamin Lee
  pub_count: 3
  first_year: 2020
  last_year: 2023
  career_length: 4
  keywords: {'augmented reality,immersive analytics,situated visualization,design patterns,design space': 1, 'immersive analytics,collaboration,virtual reality,qualitative study,multivariate data': 1, 'data visceralization,virtual reality,exploratory study': 1}
  topic_0: 1
  topic_1: 1
  topic_2: 1
  degree: 8
  clustering: 0.5
  betweenness: 0.001115311704911144
  h_index_approx: 3


## 2. Create NodeSubstrates Widget

The initial view shows a force-directed layout revealing collaboration structure.

In [20]:
# Create widget without auto-substrate
# You can manually create substrates later using lasso selection or accepting suggestions
widget = NodeSubstratesWidget(G, auto_substrate=False, width=1200, height=800)

# Interaction hints:
# - Pan: drag on empty space
# - Zoom: scroll wheel
# - Lasso select: Shift+drag (popup appears automatically)
# - Move substrate: Ctrl+drag on substrate region
# - Context menu: right-click on node or substrate
widget

Layout 'spring' computed in 7.34s for 500 nodes


## 3. Explore Auto-detected Substrate Suggestions

NodeSubstrates automatically identifies regions where attribute-based
visualization would be beneficial, based on:
1. **Cluster potential**: High attribute variance within topological communities
2. **Outlier visibility**: Nodes with unusual attribute combinations
3. **Attribute dimensionality**: Benefit from DR summarization

In [4]:
print("Auto-detected substrate suggestions:\n")
for i, suggestion in enumerate(widget.suggested_regions):
    print(f"Suggestion {i}: {suggestion['label']}")
    print(f"  Nodes: {len(suggestion['node_ids'])}")
    print(f"  Score: {suggestion['score']:.3f}")
    print(f"  Reason: {suggestion['reason']}")
    print(f"  Recommended DR: {suggestion['recommended_dr']}")
    print()

Auto-detected substrate suggestions:

Suggestion 0: Community 3
  Nodes: 15
  Score: 0.689
  Reason: 15 nodes with high attribute diversity, potential outliers among neighbors, dense edge structure
  Recommended DR: umap

Suggestion 1: Community 10
  Nodes: 18
  Score: 0.689
  Reason: 18 nodes with high attribute diversity, potential outliers among neighbors
  Recommended DR: umap

Suggestion 2: Community 15
  Nodes: 29
  Score: 0.643
  Reason: 29 nodes with moderate attribute diversity, similar nodes scattered in topology, potential outliers among neighbors
  Recommended DR: umap



## 4. Accept a Suggestion

Let's accept the first suggestion to see the substrate view.

In [5]:
if widget.suggested_regions:
    substrate_id = widget.accept_suggestion(0)
    print(f"Created substrate: {substrate_id}")
    print(f"Method: {widget.substrates[0]['dr_method']}")
else:
    print("No suggestions available")

Created substrate: substrate_1
Method: umap


/home/christinoleo/Projects/papers/nodeSubstrates/implementation/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/christinoleo/Projects/papers/nodeSubstrates/implementation/.venv/lib/python3.13/site-packages/umap/umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(


## 5. Compare DR Methods

Try different dimensionality reduction methods to see different patterns:
- **PCA**: Interpretable axes (e.g., "PC1 correlates with seniority")
- **UMAP**: Reveals clusters effectively
- **t-SNE**: Emphasizes local structure

In [6]:
# Switch to UMAP for cluster discovery
if widget.substrates:
    widget.update_dr_method(widget.substrates[0]['id'], 'umap')
    print("Switched to UMAP - look for clusters of similar researchers")

Switched to UMAP - look for clusters of similar researchers


/home/christinoleo/Projects/papers/nodeSubstrates/implementation/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/christinoleo/Projects/papers/nodeSubstrates/implementation/.venv/lib/python3.13/site-packages/umap/umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(


In [7]:
# Switch to PCA for interpretable axes
if widget.substrates:
    widget.update_dr_method(widget.substrates[0]['id'], 'pca')
    print("Switched to PCA - axes may correlate with specific attributes")

Switched to PCA - axes may correlate with specific attributes


## 6. Create Multiple Substrates

You can have multiple substrate regions to compare different groups.

In [8]:
# Accept another suggestion if available
if len(widget.suggested_regions) > 1:
    substrate_id = widget.accept_suggestion(1)
    print(f"Created second substrate: {substrate_id}")

Created second substrate: substrate_2


/home/christinoleo/Projects/papers/nodeSubstrates/implementation/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## 7. Interactive Selection

- **Click** a node to select it
- **Shift+Click** to add/remove from selection
- **Shift+Drag** to lasso select multiple nodes

Then create a substrate from your selection:

In [9]:
# Check current selection
print(f"Selected nodes: {widget.selected_nodes}")

# Create substrate from selection (need at least 3 nodes)
if len(widget.selected_nodes) >= 3:
    substrate_id = widget.create_substrate(
        widget.selected_nodes,
        dr_method='umap',
        label='My Selection'
    )
    print(f"Created substrate from selection: {substrate_id}")

Selected nodes: []


## 8. Dissolve Substrates

Return nodes to the force-directed layout to restore topological context.

In [10]:
# List current substrates
print("Current substrates:")
for s in widget.substrates:
    print(f"  {s['id']}: {s['label']} ({len(s['node_ids'])} nodes, {s['dr_method']})")

Current substrates:
  substrate_0: Community 3 (15 nodes, pca)
  substrate_1: Community 3 (15 nodes, umap)
  substrate_2: Community 10 (18 nodes, umap)


In [11]:
# Dissolve first substrate
if widget.substrates:
    substrate_to_dissolve = widget.substrates[0]['id']
    widget.dissolve_substrate(substrate_to_dissolve)
    print(f"Dissolved {substrate_to_dissolve}")

Dissolved substrate_0


## 9. Understanding the Hybrid View

NodeSubstrates enables fluid exploration between:

- **Force-directed view**: Shows collaboration structure (topology)
- **Substrate view**: Shows attribute similarity (DR projection)

Edges connect across both representations:
- **Straight edges**: Within same representation type
- **Curved edges**: Cross-representation (substrate ↔ force-directed)

This hybrid approach reveals patterns that neither view alone would expose:
- Sub-communities within dense collaboration clusters
- Bridging researchers connecting different topic areas
- Outliers with unusual attribute combinations

In [12]:
# Summary statistics
print(f"\n=== NodeSubstrates Summary ===")
print(f"Total nodes: {len(widget.nodes)}")
print(f"Nodes in substrates: {len(widget.substrate_node_ids)}")
print(f"Nodes in force-directed: {len(widget.topological_node_ids)}")
print(f"Active substrates: {len(widget.substrates)}")


=== NodeSubstrates Summary ===
Total nodes: 150
Nodes in substrates: 33
Nodes in force-directed: 117
Active substrates: 2
